# Using Causal Reasoning to Test Scientific Software

## Michael Foster and Sylvia Whittle

m.foster@sheffield.ac.uk &nbsp;&nbsp;&nbsp;&nbsp;&nbsp; sylvia.whittle@sheffield.ac.uk

# Motivating Example: Covasim
- Stochastic agent-based simulator for performing COVID-19 analyses
- Used to inform research studies and policy decisions in multiple countries including the US, UK, and Australia
- Important to make sure the model is working properly

In [ ]:
import covasim as cv
sim = cv.Sim(location="UK", beta=0.2, verbose=0)
sim.run()
sim.summarize()

# A basic regression test

In [ ]:
def test_run_uk():
    sim = cv.Sim(location="UK", beta=0.2, verbose=0)
    sim.run()
    final_infections = sim.results['cum_infections'][-1]
    expected_value = 48289
    assert final_infections == expected_value, f"{final_infections} != expected_value"
    print(f"{final_infections} == {expected_value}")
test_run_uk()

# Is the model working correctly?
If it is, we should see 48289 every time we run the simulator...

In [ ]:
msim = cv.MultiSim(cv.Sim(location="UK", beta=0.016, verbose=0))
msim.run(n_runs=5)
for i, sim in enumerate(msim.sims):
    print(f"Run {i+1}: {sim.results['cum_infections'][-1]:,.0f} total infections")

**So which is correct?**

# Naive solution: Adding a tolerance

Instead of asserting `output == expected_value`, assert output is _approximately_ the expected value.
- Choosing too small a tolerance will lead to failing tests
- Choosint too large a tolerance will lead to a weak test

# What happens with a more infections variant?

In [ ]:
msim = cv.MultiSim(cv.Sim(location="UK", beta=0.017, verbose=0))
msim.run(n_runs=5)
for i, sim in enumerate(msim.sims):
    print(f"Run {i+1}: {sim.results['cum_infections'][-1]:,.0f} total infections")

**Are these results what we'd expect?**

# Metamorphic testing

Instead of asserting that `f(x) = y`, assert that _changing_ `x` leads to a _corresponding change in_ `y`.

`beta_1 > beta_2 ==> cum_infections_1 > cum_infections_2`

In [ ]:
msim_1 = cv.MultiSim(cv.Sim(location="UK", beta=0.016, verbose=0, pop_type="hybrid"))
msim_2 = cv.MultiSim(cv.Sim(location="UK", beta=0.017, verbose=0, pop_type="hybrid"))

msim_1.run(n_runs=5)
msim_2.run(n_runs=5)
for sim1, sim2 in zip(msim_1.sims, msim_2.sims):
    cum_infections_1 = sim1.results['cum_infections'][-1]
    cum_infections_2 = sim2.results['cum_infections'][-1]
    assert cum_infections_2 > cum_infections_1, f"{cum_infections_2} was less than {cum_infections_1}"

**Does this mean that there's a problem?**

Not necessarily - remember covasim is inherently stochastic

# Statistical metamorphic testing

Instead of testing relationships between exact values, test relationships between _populatiuons_ of runs.

In [ ]:
import numpy as np

def test_population_means(n_runs: int):
    msim_1 = cv.MultiSim(cv.Sim(location="UK", beta=0.016, verbose=0, pop_type="hybrid"))
    msim_2 = cv.MultiSim(cv.Sim(location="UK", beta=0.017, verbose=0, pop_type="hybrid"))
    
    msim_1.run(n_runs=n_runs)
    msim_2.run(n_runs=n_runs)
    
    cum_infections_1 = [sim1.results['cum_infections'][-1] for sim1 in msim_1.sims]
    cum_infections_2 = [sim1.results['cum_infections'][-1] for sim1 in msim_2.sims]
    
    assert np.mean(cum_infections_2) > np.mean(cum_infections_1)

test_population_means(n_runs=5)

# Is this feasible?

Running with five repeats isn't very statistically significant. Let's try it again with 30 repeats for each.

In [ ]:
from time import time

start = time()
test_population_means(30)
end = time()
print(f"That took {end - start} seconds!")

Now let's test every pair of countries - that's [TODO - check how many countries] * 9 seconds = [TODO]!

Covasim has more than 60 parameters, many of which are complex objects with their own sub-parameters!

Testing all of the relevant relationships with non-trivial population sizes could take days!

# Causal testing

Causal testing aleviates this problem by allowing us to re-use data.

The process of data collection (running the model) is completely separate from performing the tests (performing the validation).

The [Causal Testing Framework](https://github.com/CITCOM-project/CausalTestingFramework) provides a suite of tools to facilitate causal testing.

![Causal testing framework workflow](https://github.com/CITCOM-project/CausalTestingFramework/raw/main/images/schematic.png)

# Specify the expected causal relationships

A _directed acyclic graph_ (DAG) shows the expected relationships between model parameters and outputs.7

An edge `X -> Y` represents that `X` causes `Y`, i.e. that the value of `Y` somehow depends on the value of `X`.

```
digraph expected_relationships {
    location -> cum_infections;
    beta -> cum_infections;
}
```

[TODO] Visualise this

# Covasim is actually a little more complex

```
digraph expected_relationships {
    location -> average_age;
    location -> contacts_home;
    average_age -> contacts_work;
    average_age -> contacts_school;
    average_age -> contacts_community;
    beta -> cum_infections;
    contacts_home -> cum_infections;
    contacts_school -> cum_infections;
    contacts_work -> cum_infections;
    contacts_community -> cum_infections;
}
```

The location does not determine the cumulative infections directly.

Instead, it determines the average age of the population and the number of contacts each agent has at home, school, work, and in the community.

The user has no direct control over these parameters from outside the model. They can only change the location.

# Generating causal tests

TODO: Command to generate causal tests from a DAG

Show what one causal test looks like

# Collecting data

TODO: Basically just run the model a bunch of times with random values for each parameter

# Running test cases

Put the command to run the test cases

# Visualising the results

TODO: Visualise the results

# Interpreting the results

Just because a test has failed doesn't mean there's a problem with the model! There could be a problem with our causal DAG or we may not have enough data.

# Evaluating causal DAGs

We can see how well a given causal DAG fits the data we have, and estimate the confidence that it's accurate.

# Causal Test Adequacy
TODO (if time) show command for test adequacy

# Inferring causal DAGs

We can also infer causal DAGs from the data

TODO: put the command for that and show an inferred DAG

# Conclusion

- Scientific software is inherently hard to test
- Instead of asserting that a particular input configuration results in a particular output configuration, we can test the _relationships_ between inputs and outputs.
- Collecting repeated test runs for every parameter configuration we want to test can be infeasible.
- Causal testing allows us to maximise what we can do with the test runs we are able to collect.
- The Causal Testing Framework automates much of this process.